# EEG P300 BCI Speller — Exploratory Analysis

This notebook explores the raw EEG dataset, visualises the P300 ERP, and validates the preprocessing pipeline before running the full benchmarking script.

**Run order:** Execute cells top-to-bottom. The notebook is self-contained and requires the project `src/` on `sys.path` (handled in Cell 1).

In [ ]:
import sys
from pathlib import Path

# Add src/ to path so we can import project modules
src_path = str(Path('..') / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import numpy as np
import matplotlib.pyplot as plt
import mne
mne.set_log_level('WARNING')
print('Environment ready.')

## 1. Load Dataset & Inspect Raw EEG

In [ ]:
from preprocess import get_clean_data

# Load Subject 1, BNCI2014_009 (change as needed)
DATASET = 'BNCI2014_009'
SUBJECT = 1

epochs, X, y = get_clean_data(DATASET, SUBJECT)

print(f'Epochs shape : {X.shape}  (n_epochs, n_channels, n_times)')
print(f'Label distribution: Target={y.sum()}, NonTarget={len(y)-y.sum()}')
print(f'Sampling rate : {epochs.info["sfreq"]} Hz')
print(f'Channels       : {len(epochs.ch_names)}')
epochs.info

## 2. P300 Grand Average ERP (Target vs Non-Target)

In [ ]:
# Grand average across all target and non-target epochs
target_epochs = epochs[y == 1]
nontarget_epochs = epochs[y == 0]

target_ga = target_epochs.get_data().mean(axis=0)       # (n_ch, n_times)
nontarget_ga = nontarget_epochs.get_data().mean(axis=0)

times = epochs.times * 1000  # convert to ms

# Plot grand average for Pz (canonical P300 channel)
pz_idx = epochs.ch_names.index('Pz') if 'Pz' in epochs.ch_names else 0

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(times, target_ga[pz_idx] * 1e6, label='Target (P300)', color='red', lw=2)
ax.plot(times, nontarget_ga[pz_idx] * 1e6, label='Non-Target', color='blue', lw=2, alpha=0.7)
ax.axvline(0, color='k', linestyle='--', lw=1, label='Stimulus onset')
ax.axvline(300, color='green', linestyle=':', lw=1, label='~300 ms (P300 peak)')
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Amplitude (µV)')
ax.set_title(f'Grand Average ERP — {DATASET} Subject {SUBJECT} (Channel: {epochs.ch_names[pz_idx]})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/grand_average_erp.png', dpi=150)
plt.show()
print('ERP plot saved to results/grand_average_erp.png')

## 3. Topographic Map at P300 Peak (~300 ms)

In [ ]:
# Difference wave: Target minus NonTarget
diff_evoked = mne.combine_evoked(
    [target_epochs.average(), nontarget_epochs.average()],
    weights=[1, -1]
)

fig = diff_evoked.plot_topomap(times=[0.1, 0.2, 0.3, 0.4, 0.5], ch_type='eeg', show=False)
fig.suptitle('Topomap: Target − NonTarget Difference Wave', y=1.05)
plt.tight_layout()
plt.savefig('../results/p300_topomap.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Epoch Statistics & Data Quality Check

In [ ]:
import pandas as pd

# Channel-wise SNR: ratio of target ERP peak to noise floor
peak_window = (times >= 200) & (times <= 500)
baseline_window = (times >= -200) & (times < 0)

snr_values = []
for ch_idx in range(len(epochs.ch_names)):
    signal = np.abs(target_ga[ch_idx, peak_window]).max()
    noise = target_ga[ch_idx, baseline_window].std() + 1e-9
    snr_values.append({'channel': epochs.ch_names[ch_idx], 'SNR': signal / noise})

snr_df = pd.DataFrame(snr_values).sort_values('SNR', ascending=False)
print('Top 10 channels by P300 SNR:')
print(snr_df.head(10).to_string(index=False))

# Basic epoch count check
print(f'\nTotal epochs : {len(epochs)}')
print(f'Target       : {y.sum()} ({100*y.mean():.1f}%)')
print(f'Non-target   : {(y==0).sum()} ({100*(1-y.mean()):.1f}%)')

## 5. Feature Extraction Preview

In [ ]:
from features import extract_p300_features, apply_xdawn

# Downsampled waveform features (decimation=3)
X_flat = extract_p300_features(X, decimation_factor=3)
print(f'Flat feature shape (dec=3): {X_flat.shape}')
print(f'  => {X.shape[1]} channels × {X.shape[2]//3} time points = {X_flat.shape[1]} features')

# Xdawn features (train/test split for preview)
split = int(0.8 * len(epochs))
X_tr_xd, X_te_xd = apply_xdawn(epochs[:split], y[:split], epochs[split:], decimation_factor=3)
print(f'Xdawn feature shape (train): {X_tr_xd.shape}')
print(f'Xdawn feature shape (test) : {X_te_xd.shape}')

## ✅ Exploration Complete

The plots confirm:
- A clear P300 deflection in the **Target** ERP at ~300 ms
- Posterior channels (Pz, Cz) show the highest SNR as expected for P300
- Epoch counts and label distribution look healthy

Proceed to `python src/evaluate.py` for the full benchmarking run.